# # Load training data

In [1]:
# Demo scaffolding — replace with your own data source in production.
from src.iris_demo import prepare
conn_name = prepare()

from nubison_model import data

db = data.connection(conn_name)
df = db.load("SELECT * FROM iris")
datasets = data.split(df, {"training": 0.8, "evaluation": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:12s} rows={len(sub):3d}  source_uri={sub.attrs['source_uri']}")


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


training     rows=120  source_uri=dbref://IRIS#a295df6f/training

evaluation   rows= 30  source_uri=dbref://IRIS#a295df6f/evaluation

# # Train (PyTorch)

In [2]:
# `train()` yields a `TrainContext` that exposes data + mlflow-wrapped
# helpers. For PyTorch you write the learning loop directly and call
# `t.log_metric` / `t.save` — no `import mlflow` needed.
#   t.X, t.y           — features / target as DataFrames (from "training")
#   t.log_metric(...)  — wraps mlflow.log_metric
#   t.save(model)      — pickle + log_artifact to `weights_path`
import numpy as np
import torch
import torch.nn as nn
from nubison_model import train

WEIGHTS_PATH = "src/weights.pkl"


class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))

    def forward(self, x):
        return self.fc(x)


with train(datasets=datasets, target="target", weights_path=WEIGHTS_PATH) as t:
    X = torch.tensor(t.X.values, dtype=torch.float32)
    y = torch.tensor(t.y.values, dtype=torch.long)

    model = IrisNet()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(30):
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        t.log_metric("loss", loss.item(), step=epoch)

    t.save(model)

print(f"run_id: {t.run_id}")
print(f"weights: {WEIGHTS_PATH}")


2026/05/15 18:38:22 INFO mlflow.tracking.fluent: Experiment with name 'PyTorch_1778837897' does not exist. Creating a new experiment.


2026/05/15 18:38:23 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run skittish-goose-274 at: http://127.0.0.1:5050/#/experiments/6/runs/c86082de053c495ea5a2beebea26a0f0


🧪 View experiment at: http://127.0.0.1:5050/#/experiments/6


run_id: c86082de053c495ea5a2beebea26a0f0

weights: src/weights.pkl

# # Next: package for inference

The trained model is now pickled at `src/weights.pkl`. To serve it as a
REST endpoint, see `model.ipynb` in this same directory.